In [ ]:
from pathlib import Path
import os
import subprocess
import sys

from google.colab import drive


def run(cmd, cwd=None):
    print("Running:", " ".join(map(str, cmd)))
    proc = subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert proc.stdout is not None
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if code:
        raise RuntimeError(
            f"Command failed with exit code {code}: "
            + " ".join(map(str, cmd))
        )


drive.mount("/content/drive")

WORKDIR = Path("/content/drive/MyDrive/Research/SSM_World_Models")
R2DREAMER_DIR = WORKDIR / "r2dreamer"
TDMPC2_DIR = WORKDIR / "tdmpc2"
DATA_DIR = WORKDIR / "data" / "dmc_expert"

R2DREAMER_REPO = "https://github.com/hanswalker/r2dreamer.git"
R2DREAMER_BRANCH = "mamba3-rssm-experiment"
TDMPC2_REPO = "https://github.com/nicklashansen/tdmpc2.git"

WORKDIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Colab starts with only this notebook. Clone the R2Dreamer repo that contains
# scripts/collect_dmc_expert_data.py, or update the Drive clone from the same
# GitHub source even if its default origin points somewhere else.
if not R2DREAMER_DIR.exists():
    run([
        "git",
        "clone",
        "--branch",
        R2DREAMER_BRANCH,
        R2DREAMER_REPO,
        str(R2DREAMER_DIR),
    ])
else:
    remotes = subprocess.run(
        ["git", "remote"],
        cwd=R2DREAMER_DIR,
        check=True,
        capture_output=True,
        text=True,
    ).stdout.split()
    if "notebook-source" in remotes:
        run(["git", "remote", "set-url", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    else:
        run(["git", "remote", "add", "notebook-source", R2DREAMER_REPO], cwd=R2DREAMER_DIR)
    run(["git", "fetch", "notebook-source", R2DREAMER_BRANCH], cwd=R2DREAMER_DIR)
    run(["git", "checkout", "-B", R2DREAMER_BRANCH, f"notebook-source/{R2DREAMER_BRANCH}"], cwd=R2DREAMER_DIR)

# TD-MPC2 is the expert policy repo. The collector imports its model code and
# downloads its released checkpoints from Hugging Face.
if not TDMPC2_DIR.exists():
    run(["git", "clone", TDMPC2_REPO, str(TDMPC2_DIR)])
else:
    run(["git", "pull", "--ff-only"], cwd=TDMPC2_DIR)

run([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "torch",
    "numpy==1.26.0",
    "pyyaml",
    "zarr<3",
    "huggingface_hub",
    "dm_control==1.0.28",
    "mujoco==3.3.0",
    "omegaconf",
    "hydra-core",
    "tensordict",
    "torchrl",
    "kornia",
    "termcolor",
    "tqdm",
    "pandas",
    "moviepy",
    "imageio",
    "imageio-ffmpeg",
    "h5py",
])

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TDMPC2_DIR"] = str(TDMPC2_DIR)
os.environ["DMC_EXPERT_DATA_DIR"] = str(DATA_DIR)

os.chdir(R2DREAMER_DIR)
if str(R2DREAMER_DIR) not in sys.path:
    sys.path.insert(0, str(R2DREAMER_DIR))

collector_candidates = [
    R2DREAMER_DIR / "scripts" / "collect_dmc_expert_data.py",
    R2DREAMER_DIR / "collect_dmc_expert_data.py",
]
COLLECTOR = next((path for path in collector_candidates if path.exists()), None)
if COLLECTOR is None:
    raise FileNotFoundError(
        "Missing collect_dmc_expert_data.py in the cloned R2Dreamer repo. "
        f"Checked: {collector_candidates}"
    )

R2DREAMER_COMMIT = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"],
    cwd=R2DREAMER_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()

import torch

print("R2Dreamer:", R2DREAMER_DIR)
print("R2Dreamer source:", R2DREAMER_REPO, R2DREAMER_BRANCH, R2DREAMER_COMMIT)
print("TD-MPC2:", TDMPC2_DIR)
print("Data:", DATA_DIR)
print("Collector:", COLLECTOR)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        "TD-MPC2 requires CUDA. In Colab, choose Runtime > Change runtime type > GPU, "
        "then rerun the notebook."
    )


In [ ]:
CONFIG_PATH = R2DREAMER_DIR / "configs" / "dmc_expert_small_train.yaml"

print("Config:", CONFIG_PATH)
print(CONFIG_PATH.read_text())
run([sys.executable, str(COLLECTOR), "--help"])
run([sys.executable, str(COLLECTOR), "--config", str(CONFIG_PATH)])
